In [ ]:
##
import os
import sqlite3
import pandas as pd

db_path = "../data/raw/MIG_Cement_Records.db"
print("Database found:", os.path.exists(db_path))

Database found: True


In [ ]:
##Connect to the database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [ ]:
#List the tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
tables

[('Sites',), ('CementTypes',), ('Operations',)]

In [ ]:
##Show each table’s columns
for table_name, in tables:
    print(f"===== {table_name} =====")
    cursor.execute(f"PRAGMA table_info({table_name});")
    for column in cursor.fetchall():
        print(column[1], "-", column[2])
    print()

===== Sites =====
site_id - TEXT
region - TEXT
silo_capacity - INTEGER
behavior - TEXT

===== CementTypes =====
cement_type - TEXT

===== Operations =====
date - TEXT
site_id - TEXT
cement_type - TEXT
planned_pour_tonnes - REAL
consumed_tonnes - REAL
opening_inventory_tonnes - REAL
deliveries_tonnes - REAL
closing_inventory_tonnes - REAL
rain_mm - REAL
avg_temp_c - REAL
silo_capacity - INTEGER



In [ ]:
##Join the tables and load them into pandas
query = """
    SELECT
        o.date, o.site_id, s.region, s.behavior, o.cement_type,
        o.planned_pour_tonnes, o.consumed_tonnes, o.opening_inventory_tonnes,
        o.deliveries_tonnes, o.closing_inventory_tonnes, o.rain_mm,
        o.avg_temp_c, o.silo_capacity
    FROM Operations o
    JOIN Sites s ON o.site_id = s.site_id
"""

df = pd.read_sql_query(query, conn)
df.head()

In [ ]:
#Check the result
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      32880 non-null  str    
 1   site_id                   32880 non-null  str    
 2   region                    32880 non-null  str    
 3   behavior                  32880 non-null  str    
 4   cement_type               32880 non-null  str    
 5   planned_pour_tonnes       32880 non-null  float64
 6   consumed_tonnes           32880 non-null  float64
 7   opening_inventory_tonnes  32880 non-null  float64
 8   deliveries_tonnes         32880 non-null  float64
 9   closing_inventory_tonnes  32880 non-null  float64
 10  rain_mm                   32880 non-null  float64
 11  avg_temp_c                32880 non-null  float64
 12  silo_capacity             32880 non-null  int64  
dtypes: float64(7), int64(1), str(5)
memory usage: 3.3 MB


In [7]:
#Close the connection is a good practice to close the database once you’ve loaded what you need.
conn.close()